## 1. Setup & Data Loading

In [1]:
!pip install autogluon

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached gdown-5.2.0-py3-none-any.whl.metadata (5.8 kB)
  Preparing metadata (setup.py) ... done
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
INFO: pip is looking at multiple versions of huggingface-hub[torch] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of huggingface-hub[torch] to determine which version is compatible with other requirements. This could take a while.
  Using cached hf_xet-1.2.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.9 kB)
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py) ... done
INFO: pip is looking a

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# AutoGluon
from autogluon.tabular import TabularDataset, TabularPredictor

In [3]:
DATA_PATH = Path("/home/jovyan/__DATA/APBDID_F25/data/handm")

In [4]:
# Load datasets
articles_df = pd.read_csv(DATA_PATH / "articles.csv")
customers_df = pd.read_csv(DATA_PATH / "customers.csv")
transactions_df = pd.read_csv(DATA_PATH / "transactions_train.csv")

print(f"Articles: {articles_df.shape}")
print(f"Customers: {customers_df.shape}")
print(f"Transactions: {transactions_df.shape}")

Articles: (105542, 25)
Customers: (1371980, 7)
Transactions: (31788324, 5)


In [5]:
# Quick look at transactions (our main table)
transactions_df.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


## 2. Minimal Preprocessing

Keep it simple:
- Sample transactions (full dataset is too large for quick experiments)
- Merge customer and article features
- Target: `article_id` (what article was purchased)

In [6]:
# Sample for speed - use more data for better results
SAMPLE_SIZE = 50_000
np.random.seed(42)

transactions_sample = transactions_df.sample(n=SAMPLE_SIZE, random_state=42)
print(f"Sampled transactions: {len(transactions_sample):,}")

Sampled transactions: 50,000


In [ ]:
# Select ONLY the target from articles (no other article features to avoid leakage)
articles_target = articles_df[["article_id", "product_group_name"]].copy()

# Select key features from customers (these are legitimate predictors)
customers_features = customers_df[[
    "customer_id",
    "club_member_status",
    "fashion_news_frequency",
    "age"
]].copy()

# Fill missing values
customers_features["club_member_status"] = customers_features["club_member_status"].fillna("UNKNOWN")
customers_features["fashion_news_frequency"] = customers_features["fashion_news_frequency"].fillna("UNKNOWN")
customers_features["age"] = customers_features["age"].fillna(customers_features["age"].median())

In [ ]:
# Merge: transactions + customer features + target only
train_df = transactions_sample.merge(customers_features, on="customer_id", how="left")
train_df = train_df.merge(articles_target, on="article_id", how="left")

print(f"Training data shape: {train_df.shape}")
train_df.head()

Training data shape: (50000, 13)


,t_dat,customer_id,article_id,price,sales_channel_id,club_member_status,fashion_news_frequency,age,product_group_name,colour_group_name,department_name,index_group_name,garment_group_name
0,2019-09-13,215895f90002eb3d1a04bd603513c8e85e6002ef08f136...,786586001,0.022017,1,ACTIVE,NONE,41.0,Garment Lower body,Black,Young Girl Jersey Fancy,Baby/Children,Jersey Fancy
1,2019-02-23,7b183268e3a4623b80d5325ec4a20a0af0edff7bcb1748...,658911001,0.028797,2,ACTIVE,Regularly,30.0,Nightwear,Dark Blue,Nightwear,Ladieswear,"Under-, Nightwear"
2,2019-07-17,2eb7412239a90c0570cd3d1bf0492856ae5b59058b1ea6...,759326005,0.050831,2,ACTIVE,Regularly,22.0,Swimwear,Dark Orange,Swimwear,Ladieswear,Swimwear
3,2019-05-16,74f162e5a170fd57207aa2a7d5c58479ee9de903b2a277...,737137004,0.027102,1,ACTIVE,NONE,21.0,Garment Upper body,Dark Green,Blouse,Ladieswear,Blouses
4,2019-08-10,aab9306ee28c4db494003955f80355e540b01480ab35cf...,785931001,0.050831,2,ACTIVE,Regularly,49.0,Garment Lower body,White,Trouser,Ladieswear,Trousers


In [9]:
# Simplify target: predict product_group_name instead of exact article_id
# (article_id has too many unique values for quick training)
TARGET = "product_group_name"

print(f"Target classes: {train_df[TARGET].nunique()}")
train_df[TARGET].value_counts()

Target classes: 15


product_group_name
Garment Upper body       19801
Garment Lower body       11086
Garment Full body         5502
Underwear                 4036
Swimwear                  4032
Accessories               2577
Shoes                     1147
Socks & Tights            1060
Nightwear                  585
Unknown                    159
Items                        7
Bags                         4
Cosmetic                     2
Garment and Shoe care        1
Furniture                    1
Name: count, dtype: int64

In [ ]:
# Features for training (drop IDs and non-predictive columns)
drop_cols = ["customer_id", "article_id", "t_dat"]
feature_cols = [c for c in train_df.columns if c not in drop_cols and c != TARGET]

train_data = train_df[feature_cols + [TARGET]].copy()
print(f"Features: {feature_cols}")
print(f"Final training shape: {train_data.shape}")

Features: ['club_member_status', 'fashion_news_frequency', 'age', 'colour_group_name', 'department_name', 'index_group_name', 'garment_group_name']
Final training shape: (50000, 8)


## 3. Train AutoGluon Model

AutoGluon handles:
- Automatic feature engineering
- Model selection & ensembling
- Hyperparameter tuning

In [11]:
# Convert to AutoGluon dataset
train_ag = TabularDataset(train_data)

In [12]:
# Train model with time limit (quick baseline)
predictor = TabularPredictor(
    label=TARGET,
    eval_metric="accuracy",
    path="models/autogluon_baseline"
).fit(
    train_ag,
    time_limit=120,  # 2 minutes for quick demo
    presets="medium_quality"  # Options: best_quality, high_quality, medium_quality, optimize_for_deployment
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.7
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #216-Ubuntu SMP Thu Aug 29 13:26:53 UTC 2024
CPU Count:          2
Memory Avail:       22.71 GB / 31.35 GB (72.4%)
Disk Space Avail:   5.47 GB / 11.71 GB (46.7%)
	We recommend a minimum available disk space of 10 GB, and large datasets may require more.
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 120s
AutoGluon will save models to "/home/jovyan/HM-fashion-recs/models/autogluon_baseline"
Train Data Rows:    50000
Train Data Columns: 7
Label Column:       product_group_name
AutoGluon infers your prediction problem is: 'multiclass' (because dtype of label-column == object).
	First 10 (of 15) unique label values:  ['Garment Lower body', 'Nightwear', 'Swimwear', 'Garment Upper body', 'Under

## 4. Evaluate Results

In [13]:
# Model leaderboard
predictor.leaderboard(silent=True)

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBMXT,0.9024,accuracy,0.101251,4.120904,0.101251,4.120904,1,True,2
1,WeightedEnsemble_L2,0.9024,accuracy,0.102450,4.244050,0.001199,0.123146,2,True,7
2,NeuralNetFastAI,0.9000,accuracy,0.061151,40.674366,0.061151,40.674366,1,True,1
3,LightGBM,0.8996,accuracy,0.038558,2.296999,0.038558,2.296999,1,True,3
4,CatBoost,0.8988,accuracy,0.006731,59.332268,0.006731,59.332268,1,True,6
5,RandomForestEntr,0.8828,accuracy,0.116534,5.779069,0.116534,5.779069,1,True,5
6,RandomForestGini,0.8816,accuracy,0.118231,5.279549,0.118231,5.279549,1,True,4


In [14]:
# Feature importance
predictor.feature_importance(train_ag)

Computing feature importance via permutation shuffling for 7 features using 5000 rows with 5 shuffle sets...
	8.95s	= Expected runtime (1.79s per shuffle set)
	8.47s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
garment_group_name,0.38812,0.006918,1.210413e-08,5,0.402363,0.373877
department_name,0.23520,0.005081,2.612608e-08,5,0.245663,0.224737
colour_group_name,0.00604,0.000817,3.926967e-05,5,0.007723,0.004357
index_group_name,0.00264,0.001187,3.812348e-03,5,0.005083,0.000197
age,0.00120,0.000616,6.064930e-03,5,0.002469,-0.000069
fashion_news_frequency,0.00068,0.000782,6.192842e-02,5,0.002291,-0.000931
club_member_status,0.00032,0.000335,4.965034e-02,5,0.001009,-0.000369


In [15]:
# Quick prediction example
sample_customer = train_data.drop(columns=[TARGET]).iloc[:5]
predictions = predictor.predict(sample_customer)
print("Predictions:")
print(predictions)

Predictions:
0    Garment Lower body
1             Nightwear
2              Swimwear
3    Garment Upper body
4    Garment Lower body
Name: product_group_name, dtype: object


## 5. Next Steps

To improve the model:
1. **More data**: Increase `SAMPLE_SIZE`
2. **Better features**: Add lag features from transaction history (as shown in class)
3. **Longer training**: Increase `time_limit` or use `best_quality` preset
4. **Different target**: Try predicting `department_name` or `index_group_name`
5. **Evaluation**: Use proper train/validation/test split by time